# 0. 환경 설정

## import

In [1]:
import pandas as pd
from datetime import date, timedelta
import os

## 함수 정의

In [2]:
def daterange(start_date, end_date) :
    if start_date <= end_date : 
        return [start_date + timedelta(days=i) for i in range((end_date - start_date).days + 1)]
    else : 
        print('시작 날짜는 종료 날짜보다 클 수 없습니다.')
        return []

## 변수 관리

In [3]:
RAW_DATA_FOLDER = '../data/raw'

# 1. 사용자 컨트롤 영역 ⚠️

In [ ]:
# 데이터 분석을 위한 날짜를 설정.
# 8월 7일(금) 8일(토) 9일(일) 10일(월) 11일(화) 12(수) 13(목) 14(금)
# start_date = date(2026, 7, 24)
# end_date = date(2026, 8, 10)

start_date = date(2026, 1, 24)
end_date = date(2026, 8, 10)

# 2. 데이터 취합

In [5]:
date_list = daterange(start_date, end_date)

movie_scrap_df = pd.DataFrame()
review_scrap_df = pd.DataFrame()
movie_api_df = pd.DataFrame()

for current_date in date_list:
    target_date = current_date.strftime("%Y%m%d")

    movie_scrap_path = os.path.join(RAW_DATA_FOLDER, 'movie_info_scrap', f'movie_info_scrap_{target_date}.csv')
    review_scrap_path = os.path.join(RAW_DATA_FOLDER, 'movie_review_scrap', f'movie_review_scrap_{target_date}.csv')
    movie_api_path = os.path.join(RAW_DATA_FOLDER, 'movie_info_api', f'movie_info_api_{target_date}.csv')

    if os.path.exists(movie_scrap_path):
        try:
            movie_scrap = pd.read_csv(movie_scrap_path)
            movie_scrap_df = pd.concat([movie_scrap_df, movie_scrap], ignore_index=True)
        except Exception as e:
            print(f'[{target_date}] movie_info_scrap 읽기 실패: {e}')
    else:
        print(f'[{target_date}] movie_info_scrap 파일 없음: {movie_scrap_path}')

    if os.path.exists(review_scrap_path):
        try:
            review_scrap = pd.read_csv(review_scrap_path)
            review_scrap_df = pd.concat([review_scrap_df, review_scrap], ignore_index=True)
        except Exception as e:
            print(f'[{target_date}] movie_review_scrap 읽기 실패: {e}')
    else:
        print(f'[{target_date}] movie_review_scrap 파일 없음: {review_scrap_path}')

    if os.path.exists(movie_api_path):
        try:
            movie_api = pd.read_csv(movie_api_path)
            movie_api_df = pd.concat([movie_api_df, movie_api], ignore_index=True)
        except Exception as e:
            print(f'[{target_date}] movie_info_api 읽기 실패: {e}')
    else:
        print(f'[{target_date}] movie_info_api 파일 없음: {movie_api_path}')

[20260124] movie_info_scrap 파일 없음: ../data/raw/movie_info_scrap/movie_info_scrap_20260124.csv
[20260124] movie_review_scrap 파일 없음: ../data/raw/movie_review_scrap/movie_review_scrap_20260124.csv
[20260125] movie_info_scrap 파일 없음: ../data/raw/movie_info_scrap/movie_info_scrap_20260125.csv
[20260125] movie_review_scrap 파일 없음: ../data/raw/movie_review_scrap/movie_review_scrap_20260125.csv
[20260126] movie_info_scrap 파일 없음: ../data/raw/movie_info_scrap/movie_info_scrap_20260126.csv
[20260126] movie_review_scrap 파일 없음: ../data/raw/movie_review_scrap/movie_review_scrap_20260126.csv
[20260127] movie_info_scrap 파일 없음: ../data/raw/movie_info_scrap/movie_info_scrap_20260127.csv
[20260127] movie_review_scrap 파일 없음: ../data/raw/movie_review_scrap/movie_review_scrap_20260127.csv
[20260128] movie_info_scrap 파일 없음: ../data/raw/movie_info_scrap/movie_info_scrap_20260128.csv
[20260128] movie_review_scrap 파일 없음: ../data/raw/movie_review_scrap/movie_review_scrap_20260128.csv
[20260129] movie_info_scrap 파일

# 3. 데이터 확인

## movie_scrap

* `rank` : 순위
* `title` : 영화 제목
* `score` : 평점 (10점 만점)
* `daily_audience` : 일일 관람객 수
* `total_audience` : 누적 관람객 수
* `release_date` : 개봉일
* `href` : 링크
* `id` : 영화 ID (링크에서 발췌)
* `genre` : 장르
* `grade` : 등급
* `time` : 상영시간
* `director` : 감독
* `expert_score` : 전문가 평점 (10점 만점)
* `nation` : 국가
* `c_datetime` : 수집 날짜 및 시간
* `b_date` : 기준 날짜

In [6]:
# 데이터 정보 확인
# movie_scrap_df.info()

# 중복값 확인
movie_scrap_df.duplicated().sum()

# 결측값 확인
movie_scrap_df.isna().sum()
movie_scrap_df[movie_scrap_df['score'].isna()]
## 수집 당시 신규 개봉 및 예매, 신규 차트 진입으로 인해 관련 정보가 일부 누락된 상태
## 개봉 이후 시점까지 데이터를 추가 확보하여, 결측 대체 가능
## 현재 결측 컬럼 - socre, release_date, href, id, genre, grade, time, director, nation

# 이상치 확인
# rank 1위 ~ 10위
movie_scrap_df['rank'].unique() #[1, 2, ..., 10]
# socre 1점 ~ 10점
movie_scrap_df['score'].min() # 5.0
movie_scrap_df['score'].max() # 7.55
# 누적 관람객 수 >= 일일 관람객 수
movie_scrap_df[movie_scrap_df['total_audience'] < movie_scrap_df['daily_audience']] #8월 8일 수집된 데이터 중 오디세이, 사랑의 하츄핑 : 고래 보석의 전설 발견
movie_scrap_df[movie_scrap_df['title'] == '오디세이']
## 해당 데이터가 실시간으로 방문자 수를 산정하고 있는지? 등에 대한 수치와 관련된 정보를 파악하기 어려움.
## 일일 관객 수와 같은 정보는 API에 더 자세하고 신뢰성 있게 보유하고 있을 것으로 판단.
## 따라서 해당 데이터에서는 리뷰 데이터와의 연계 및 API 호출을 통해 파악하기 어려운 장르, 등급, 시간 등에 대한 정보를 활용한 마스터 테이블로 활용하는 것이 적합할 것으로 판단.

,rank,title,score,daily_audience,total_audience,release_date,href,id,genre,grade,time,director,expert_score,nation,c_datetime,b_date
1,2,오디세이,7.55,539180,539804,2026-08-05,/movie/info/?movie_id=62585,62585.0,"액션,모험,드라마",15세이상관람가,172분,크리스토퍼놀란,7.55,미국,2026-08-07 17:09:25,2026-08-07
11,2,오디세이,7.55,849336,539804,2026-08-05,/movie/info/?movie_id=62585,62585.0,"액션,모험,드라마",15세이상관람가,172분,크리스토퍼놀란,7.55,미국,2026-08-08 09:37:17,2026-08-08
21,2,오디세이,7.55,1374672,1375296,2026-08-05,/movie/info/?movie_id=62585,62585.0,"액션,모험,드라마",15세이상관람가,172분,크리스토퍼놀란,7.55,미국,2026-08-09 10:16:57,2026-08-09
31,2,오디세이,7.55,1871575,1872877,2026-08-05,/movie/info/?movie_id=62585,62585.0,"액션,모험,드라마",15세이상관람가,172분,크리스토퍼놀란,7.55,미국,2026-08-10 13:06:14,2026-08-10


## review_scrap

* `id` : 영화 ID
* `reviewr_name` : 리뷰어 이름
* `score` : 점수
* `review` : 리뷰 내용
* `c_datetime` : 수집 날짜 및 시간
* `b_date` : 기준 날짜

In [7]:
# 데이터 정보 확인
# movie_scrap_df.info()

# 중복값 확인
review_scrap_df.duplicated().sum()
review_scrap_df = review_scrap_df[['id', 'reviewer_name', 'score', 'review']].drop_duplicates().reset_index(drop=True)

# 결측값 확인
review_scrap_df.isna().sum() # 0

# 이상치 확인
# socre 1점 ~ 10점
review_scrap_df['score'].min() # 4
review_scrap_df['score'].max() # 9

np.int64(9)

In [8]:
review_scrap_df

,id,reviewer_name,score,review
0,63091,송경원,7,하이틴의 겉옷을 벗고 근본으로 돌아간 성장
1,63091,정재현,7,큰 힘과 큰 책임이 서로를 따르지 못할 때에도 당신이 히어로라면
2,63091,이자연,7,도파민형 영웅이 가득한 세상에서 ‘선함’을 지켜낸 ‘강한’ 히어로
3,63091,이우빈,7,역시 스파이더맨은 고통받을수록 강하고 재밌다
4,63091,박평식,7,캐릭터도 캐스트도 발맞춰 잘 자랐구나
5,63091,이주현,9,역대 가장 성숙한 스파이더맨의 탄생
6,63091,이용철,7,"쑥 성장한 캐릭터, 따뜻하게 감싸는 드라마"
7,63091,유선아,7,샘 레이미의 유산마저 물려받아 진화한 멜로드라마와 역동의 좋은 균형
8,63091,김현수,8,아마도 마블의 새로운 미래
9,62585,이자연,8,그리스신화는 놀런을 만나길 오랫동안 기다려왔구나


## movie_api

* `rank` : 순위
* `rankInten` : 이전일 대비 순위 증감분
* `rankOldAndNew` : 이전일 대비 랭크 신규 진입 여부 (OLD : 기존 / NEW : 신규)
* `movieCd` : 영화 대표 코드
* `movieNm` : 영화 이름 (국문)
* `openDt` : 영화 개봉일
* `salesAmt` : 해당 날짜의 매출액
* `salesShare` : 해당일자 상영작의 매출총액 대비 해당 영화의 매출비율
* `salesInten` : 전일 대비 매출액 증감분
* `salesChange` : 전일 대비 매출 증감 비율
* `salesAcc` : 누적 매출액
* `audiCnt` : 해당일의 관객수
* `audiInten` : 전일 대비 관객수 증감분
* `audiChange` : 전일 대비 관객수 증감 비율
* `audiAcc` : 누적관객수
* `scrnCnt` : 해당 일자에 상영한 스크린 수 출력
* `showCnt` : 해당 일자에 상영된 횟수 출력

In [9]:
# 데이터 정보 확인
movie_api_df.info()

# 중복값 확인
movie_api_df.duplicated().sum()

# 결측값 확인
movie_api_df.isna().sum() # 0

# 이상치 확인
#  누적 매출액 >= 해당일 매출액
movie_api_df[movie_api_df['salesAcc'] < movie_api_df['salesAmt']]
# 누적 관객수 >= 해당일 관객수
movie_api_df[movie_api_df['audiAcc'] < movie_api_df['audiCnt']]
# 상영 횟수 >= 상영 스크린 수
movie_api_df[movie_api_df['showCnt'] < movie_api_df['scrnCnt']]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1990 entries, 0 to 1989
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   rank           1990 non-null   int64  
 1   rankInten      1990 non-null   int64  
 2   rankOldAndNew  1990 non-null   object 
 3   movieCd        1990 non-null   int64  
 4   movieNm        1990 non-null   object 
 5   openDt         1990 non-null   object 
 6   salesAmt       1990 non-null   int64  
 7   salesShare     1990 non-null   float64
 8   salesInten     1990 non-null   int64  
 9   salesChange    1990 non-null   float64
 10  salesAcc       1990 non-null   int64  
 11  audiCnt        1990 non-null   int64  
 12  audiInten      1990 non-null   int64  
 13  audiChange     1990 non-null   float64
 14  audiAcc        1990 non-null   int64  
 15  scrnCnt        1990 non-null   int64  
 16  showCnt        1990 non-null   int64  
 17  b_date         1990 non-null   object 
 18  c_date  

,rank,rankInten,rankOldAndNew,movieCd,movieNm,openDt,salesAmt,salesShare,salesInten,salesChange,salesAcc,audiCnt,audiInten,audiChange,audiAcc,scrnCnt,showCnt,b_date,c_date


* `movie_scrap_df`을 활용하여, 영화의 정보를 담는 `master_table`로 활용 결정
  * 일일 관객 수가 누적 관객보다 많은 현상 발견
  * 데이터 수집 기간 중 주말이 포함되어 있었기에, 집계 방법에 따라 일일 관객수가 누적 관객수를 넘어가는 현상이 발견될 수 있다고는 생각.
  * 다만 그 데이터가 일일 관객이 어제를 기준으로 한 것인지?, 실시간인지?, 누적 관객은 어제를 기준으로 한 것인지? 
  * 명확한 판단을 하지 못한 상태에서 사용 불가능
  * 또한 해당 정보는 API를 통해서도 확인이 가능

## movie_master_df

* 스크래핑 데이터와 API 데이터를 연결할 수 있는 매개로 사용
* API 데이터를 통해 확인하기 어려운 내용 확보

In [10]:
# 마스터 테이블에 조금 더 필요한 정보로 축약
movie_master_df = movie_scrap_df[['title', 'score', 'id', 'genre', 'grade', 'time', 'director', 'expert_score', 'nation']]

# 중복을 제거 진행
# 1. 정렬: title 기준 + id가 NaN이 아닌 것(값이 있는 것)을 아래로 내림
# Na_position='first'를 주면 NaN이 위로 가고, 유효한 id가 아래로 내려갑니다.
movie_master_df = movie_master_df.sort_values(
    by=['title', 'id'], 
    na_position='first'
)

# 2. title 기준으로 중복 제거하되, 가장 아래(id가 채워진 최신 정보)를 남김
# (만약 평점이 최신인 것을 남기고 싶다면 정렬 조건을 추가 조정)
movie_master_df = movie_master_df.drop_duplicates(
    subset=['title'], 
    keep='last'
).reset_index(drop=True)

In [11]:
# 스크래핑 테이블과 API 테이블의 연결을 위한 컬럼 생성
movie_master_df = movie_master_df.merge(
    movie_api_df[['movieCd', 'movieNm']].drop_duplicates(subset=['movieNm'], keep='first'),
    left_on='title',
    right_on='movieNm',
    how='inner'
).drop(columns=['movieNm']) # 중복된 제목 컬럼(movieNm)은 삭제

In [12]:
movie_master_df

,title,score,id,genre,grade,time,director,expert_score,nation,movieCd
0,눈동자,5.00,63186.0,스릴러,15세이상관람가,105분,염지호,5.00,한국,20242402
1,다윗,5.00,63108.0,애니메이션,전체관람가,109분,필커닝햄,5.00,미국,20262902
2,명탐정 코난: 하이웨이의 타천사,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20264635
3,모아나,5.67,62818.0,"어드벤처,액션",전체관람가,115분,토마스케일,5.67,미국,20259946
4,미니언즈 & 몬스터즈,6.50,63032.0,애니메이션,전체관람가,89분,피에르코팽,6.50,미국,20261784
5,사랑의 하츄핑: 고래보석의 전설,7.00,63075.0,애니메이션,전체관람가,105분,김수훈,7.00,한국,20262381
6,스파이더맨: 브랜드 뉴 데이,7.33,63091.0,"판타지,어드벤처,액션",12세이상관람가,144분,데스틴크리튼,7.33,미국,20262770
7,어떻게 해야 했을까?,6.50,63282.0,다큐멘터리,12세이상관람가,101분,후지노토모아키,6.50,일본,20264148
8,오디세이,7.55,62585.0,"액션,모험,드라마",15세이상관람가,172분,크리스토퍼놀란,7.55,미국,20250654
9,오케이 마담2,5.00,63240.0,코미디,15세이상관람가,108분,이철하,5.00,한국,20255484


# 4. 데이터 추출

In [13]:
pd.DataFrame(movie_api_df).to_csv(f'../data/pre_processed/movie_info_{start_date.strftime("%Y%m%d")}~{end_date.strftime("%Y%m%d")}.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(review_scrap_df).to_csv(f'../data/pre_processed/review_{start_date.strftime("%Y%m%d")}~{end_date.strftime("%Y%m%d")}.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(movie_master_df).to_csv(f'../data/pre_processed/movie_master_{start_date.strftime("%Y%m%d")}~{end_date.strftime("%Y%m%d")}.csv', index=False, encoding='utf-8-sig')